# Feature Extraction Analysis – Brand Genome Engine

This notebook loads the processed features from `data/processed/features.parquet`,
visualises key feature distributions, inspects per-brand topics, and demonstrates
competitor retrieval via the FAISS / sklearn nearest-neighbour index.

> **Dependencies:** only `pandas`, `numpy`, `matplotlib` (all in `requirements.txt`).

## 1. Load Processed Features

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure project root is on sys.path so `src.*` imports work
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Load features ────────────────────────────────────────────────────────
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features.parquet"
df = pd.read_parquet(FEATURES_PATH)

print(f"Loaded {len(df)} rows × {len(df.columns)} columns from {FEATURES_PATH.name}")
print(f"Brands: {df['brand_name'].unique().tolist()}")
print(f"Rows per brand: {df.groupby('brand_name').size().to_dict()}")
df.head()

## 2. Feature Distribution Plots

Histograms and box plots for **sentiment**, **formality**, and **readability (Flesch Reading Ease)**.

In [ ]:
import matplotlib.pyplot as plt

FEATURES = ["sentiment", "formality", "readability_flesch"]
LABELS = ["Sentiment [-1, 1]", "Formality [0, 1]", "Flesch Reading Ease"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# ── Row 1: Histograms ────────────────────────────────────────────────────
for ax, feat, label in zip(axes[0], FEATURES, LABELS):
    ax.hist(df[feat].dropna(), bins=15, edgecolor="white", alpha=0.85, color="#4C72B0")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(f"{label} – Distribution")

# ── Row 2: Box plots by brand ────────────────────────────────────────────
brands = sorted(df["brand_name"].unique())
for ax, feat, label in zip(axes[1], FEATURES, LABELS):
    data = [df[df["brand_name"] == b][feat].dropna().values for b in brands]
    ax.boxplot(data, labels=brands, vert=True)
    ax.set_ylabel(label)
    ax.set_title(f"{label} – by Brand")
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

In [ ]:
# ── Additional features ───────────────────────────────────────────────────
EXTRA_FEATURES = ["avg_sentence_length", "punctuation_density", "vocab_diversity"]
EXTRA_LABELS = ["Avg Sentence Length", "Punctuation Density", "Vocab Diversity (TTR)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat, label in zip(axes, EXTRA_FEATURES, EXTRA_LABELS):
    ax.hist(df[feat].dropna(), bins=15, edgecolor="white", alpha=0.85, color="#55A868")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(f"{label} – Distribution")

fig.tight_layout()
plt.show()

## 3. Top Topics per Brand

The topic extractor assigns 5 keyword-topics and normalised weights per text.
Below we show one representative example per brand.

In [ ]:
topic_rows = []
for brand in sorted(df["brand_name"].unique()):
    sample = df[df["brand_name"] == brand].iloc[0]
    topics = list(sample["top_topics"])
    weights = [round(float(w), 3) for w in sample["topic_weights"]]
    topic_str = ", ".join(f"{t} ({w})" for t, w in zip(topics, weights))
    topic_rows.append({"Brand": brand, "Top Topics (weight)": topic_str})

topic_df = pd.DataFrame(topic_rows)
topic_df.style.set_properties(**{"text-align": "left"}).hide(axis="index")

In [ ]:
# ── Topic weight sanity ───────────────────────────────────────────────────
tw_sums = df["topic_weights"].apply(lambda x: float(np.sum(x)))
print(f"Topic weight sums — min: {tw_sums.min():.4f},  mean: {tw_sums.mean():.4f},  max: {tw_sums.max():.4f}")
print(f"Topics per row   — all have {df['top_topics'].apply(len).unique().tolist()} topics")

## 4. Load Retrieval Index & Metadata

In [ ]:
import json

from src.benchmarking.retrieval import backend_name, load_index, query
from scripts.build_embeddings_index import _aggregate_brand_embeddings

# ── Load saved index ──────────────────────────────────────────────────────
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"

# Detect backend-appropriate file
index_path = EMBEDDINGS_DIR / "brand_profile_index.faiss"
if not index_path.exists():
    index_path = EMBEDDINGS_DIR / "brand_profile_index.pkl"

index = load_index(str(index_path))

with open(EMBEDDINGS_DIR / "metadata.json") as f:
    meta_map = json.load(f)

# ── Aggregate brand embeddings from features ──────────────────────────────
brand_embeddings, brand_meta = _aggregate_brand_embeddings(df)

print(f"Index loaded: {index.n} brands, {index.dim}-d, backend={backend_name()}")
print(f"Metadata entries: {len(meta_map)}")
print(f"Brands aggregated from features: {len(brand_meta)}")

## 5. Competitor Query Demo

Query the index for the brand **Omega** and display the top nearest competitors
(excluding self).

In [ ]:
QUERY_BRAND = "Omega"
K = 4  # fetch k+1 internally to allow self-exclusion

# Find query brand embedding
query_vec = None
for emb, meta in zip(brand_embeddings, brand_meta):
    if meta["brand_name"] == QUERY_BRAND:
        query_vec = emb
        break

assert query_vec is not None, f"Brand '{QUERY_BRAND}' not found"

# Query
ids, dists = query(index, query_vec, k=K + 1)

# Build results table (exclude self)
results = []
for idx, dist in zip(ids, dists):
    m = meta_map[str(idx)]
    if m["brand_name"] == QUERY_BRAND:
        continue
    results.append({
        "Rank": len(results) + 1,
        "Brand ID": m["brand_id"],
        "Brand Name": m["brand_name"],
        "Cosine Distance": round(dist, 6),
    })
    if len(results) >= K:
        break

result_df = pd.DataFrame(results)
print(f"\nTop-{K} competitors for {QUERY_BRAND}:\n")
result_df

In [ ]:
# ── Full retrieval table: top-3 competitors for every brand ───────────────
all_results = []
for emb, meta in zip(brand_embeddings, brand_meta):
    ids, dists = query(index, emb, k=4)
    competitors = []
    for idx, dist in zip(ids, dists):
        m = meta_map[str(idx)]
        if m["brand_id"] == meta["brand_id"]:
            continue
        competitors.append(f"{m['brand_name']} ({dist:.4f})")
        if len(competitors) >= 3:
            break
    all_results.append({
        "Query Brand": meta["brand_name"],
        "#1": competitors[0] if len(competitors) > 0 else "",
        "#2": competitors[1] if len(competitors) > 1 else "",
        "#3": competitors[2] if len(competitors) > 2 else "",
    })

full_df = pd.DataFrame(all_results)
print("Top-3 nearest competitors per brand (cosine distance):\n")
full_df